In [1]:
!pip install -U ultralytics opencv-python-headless


In [2]:
import yaml
import os

dataset_root = "/content/drive/MyDrive/cricket_ball_data"

data_yaml = {
    "path": dataset_root,
    "train": "train/images",
    "val": "valid/images",
    "nc": 1,
    "names": ["ball"]
}

yaml_path = os.path.join(dataset_root, "data.yaml")

with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f)

print("data.yaml created at:", yaml_path)


In [2]:
from ultralytics import YOLO
import ultralytics

# Verify environment (Python, Torch, CUDA/GPU)
ultralytics.checks()

In [3]:
model = YOLO('yolo11n.pt')   # pretrained weights


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
model.train(
    data="/content/drive/MyDrive/cricket_ball_data/data.yaml",
    epochs=100,
    imgsz=960,          # higher resolution for small ball
    batch=8,
    device=0,           # GPU
    workers=2,
    project="cricket_ball_training",
    name="yolov8_cricket_ball",
    exist_ok=True
)


In [10]:
model.val(data="/content/drive/MyDrive/cricket_ball_data/data.yaml")


In [11]:
trained_model = YOLO("/content/cricket_ball_training/yolov8_cricket_ball/train/weights/best.pt")

trained_model.predict(
    source="/content/3.mov",
    conf=0.25,
    save=True
)


In [ ]:
results = trained_model("/content/3.mov", stream=True)

ball_coords = []
frame_id = 0

for r in results:
    detected = 0   # visibility flag (default = not visible)

    if r.boxes is not None and len(r.boxes) > 0:
        for box in r.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2

            detected = 1  # ball detected in this frame
            ball_coords.append([frame_id, cx, cy, detected])
            break  # only one ball, so break

    # If ball not detected, still log the frame
    if detected == 0:
        ball_coords.append([frame_id, None, None, detected])

    frame_id += 1



In [ ]:
import pandas as pd

df = pd.DataFrame(ball_coords, columns=["frame", "x", "y"])
df.to_csv("cricket_ball_coordinates.csv", index=False)

df.head()


,frame,x,y
0,0,1102.251221,588.298584
1,0,131.790787,93.175484
2,1,1102.917969,587.897644
3,2,1102.915527,587.875366
4,3,1103.081909,588.204346


In [24]:
trajectory_points = []
MAX_POINTS = 50


In [ ]:
import cv2
model = YOLO("/content/cricket_ball_training/yolov8_cricket_ball/train/weights/best.pt")

cap = cv2.VideoCapture("/content/7.mov")

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)

out = cv2.VideoWriter(
    "output_with_trajectory.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, conf=0.25)

    for r in results:
        if r.boxes is not None and len(r.boxes) > 0:
            # assuming only one ball
            box = r.boxes[0]
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)

            # store center point
            trajectory_points.append((cx, cy))

            # limit trail length
            if len(trajectory_points) > MAX_POINTS:
                trajectory_points.pop(0)

            # draw bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(
                frame, "ball",
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                (0, 255, 0), 2
            )

    # draw trajectory path
    for i in range(1, len(trajectory_points)):
        cv2.line(
            frame,
            trajectory_points[i - 1],
            trajectory_points[i],
            (0, 0, 255),   # red color
            2
        )

    out.write(frame)

cap.release()
out.release()

print("Video saved with trajectory overlay")
